# 01 — Exploración de datos

Objetivo de negocio: anticipar qué clientes están por darse de baja para poder
intervenir antes. Objetivo estadístico: estimar P(baja | atributos del cliente)
con suficiente poder de ordenamiento como para priorizar una campaña de
retención.

Este notebook responde tres preguntas y nada más:

1. ¿Qué hay en los datos y qué problemas estructurales tienen?
2. ¿Cuán desbalanceada está la variable objetivo y qué implica eso para las métricas?
3. ¿Qué variables se asocian con el abandono y cuáles no, más allá del ruido?

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src import data as D
from src import evaluate as E
from src import interpret as I
from src import segment as S
from src.features import add_engineered_features, build_preprocessor
from src.models import build_model_zoo

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

## 1. Carga y verificación estructural

`src/data.py` es la única puerta de entrada a los CSV. Antes de mirar nada,
verifica las identidades algebraicas conocidas entre columnas.

In [ ]:
crudo = D.load_raw(C.TRAIN_CSV)
print(f"Filas: {len(crudo):,}   Columnas: {crudo.shape[1]}")
print(f"Nulos: {crudo.isna().sum().sum()}")
print(f"Filas duplicadas: {crudo.duplicated().sum()}")

for identidad, error in D.check_linear_dependencies(crudo).items():
    print(f"\n{identidad}\n   error absoluto máximo: {error:.2e}")

El error es del orden de 1e-14, es decir, cero numérico:
`Avg_Open_To_Buy` es una combinación lineal exacta de otras dos columnas. No
aporta información y sí genera problemas: infla la varianza de los coeficientes
en los modelos lineales y reparte la importancia entre columnas redundantes en
los de árboles. Se descarta.

La columna `Unnamed: 0` es el índice de fila del CSV. Usarla como predictor es
un error clásico: si el archivo estuviera ordenado por alguna variable
relacionada con el target, el modelo aprendería el orden de las filas.

In [ ]:
train = add_engineered_features(D.load_split("train"))
test = add_engineered_features(D.load_split("test"))

pd.DataFrame([D.describe_split(train, "train"), D.describe_split(test, "holdout")])

Las dos particiones tienen prácticamente la misma tasa de abandono
(16.07% y 16.04%), lo que sugiere un split aleatorio estratificado o
suficientemente grande. El holdout **no se vuelve a tocar** hasta el notebook 02,
y una sola vez.

## 2. La variable objetivo

Con 16% de positivos, un modelo que prediga "nadie se va" acierta el 84% de las
veces. Ese número es la razón por la cual accuracy no sirve acá.

In [ ]:
base = E.baseline_metrics(train[C.TARGET])
print("Modelo trivial (predecir 0 para todos):")
for k, v in base.items():
    print(f"  {k:>10}: {v:.4f}")

print("\nUn modelo con 84% de accuracy y 0% de recall no identifica")
print("un solo cliente en riesgo. La métrica de referencia será PR-AUC,")
print("cuya línea base es la prevalencia (0.16), no 0.5.")

## 3. Variables numéricas

Correlación con el target. Es lineal y bivariada, así que subestima relaciones
no monótonas y no captura interacciones; sirve como primer mapa, no como
selección de variables.

In [ ]:
num = [c for c in train.select_dtypes("number").columns if c != C.TARGET]
corr = train[num + [C.TARGET]].corr()[C.TARGET].drop(C.TARGET).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colores = ["#c0392b" if v > 0 else "#2c5f8a" for v in corr]
ax.barh([C.label(i) for i in corr.index], corr.values, color=colores, alpha=0.9)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Correlación de Pearson con abandono")
ax.set_title("Rojo: asociado a mayor riesgo. Azul: a menor riesgo.")
ax.grid(alpha=0.25, ls=":")
plt.show()

El patrón es coherente y transaccional: **transaccionar menos, hacerlo
por montos menores y desacelerar entre trimestres** se asocia al abandono, y
**contactar más al banco** también, lo que sugiere fricción antes de la baja.

Ninguna correlación supera 0.4 en valor absoluto. Sin embargo, los modelos de
árboles llegan a PR-AUC de 0.96 en el notebook 02. La brecha es informativa: el
poder predictivo está en las **interacciones**, no en los efectos marginales.

## 4. Variables categóricas

Acá conviene mirar con cuidado. La versión original de este análisis reportaba
que las tarjetas Platinum tienen una tasa de abandono de 23.5%, la más alta de
todas, y la interpretaba como un segmento premium en riesgo.

El problema: son 17 clientes y 4 bajas.

In [ ]:
def tabla_categoria(df, col):
    g = df.groupby(col)[C.TARGET].agg(conteo="size", bajas="sum", tasa="mean")
    # intervalo de confianza binomial normal al 95%
    err = 1.96 * np.sqrt(g["tasa"] * (1 - g["tasa"]) / g["conteo"])
    g["ic_inferior"] = (g["tasa"] - err).clip(0)
    g["ic_superior"] = (g["tasa"] + err).clip(upper=1)
    return g.sort_values("tasa", ascending=False).round(3)

tabla_categoria(train, "Card_Category")

In [ ]:
base_rate = train[C.TARGET].mean()

# ¿Qué categorías tienen un intervalo que NO contiene la tasa general?
# Sólo esas son distinguibles del promedio de la cartera.
for col in C.CATEGORICAL:
    g = tabla_categoria(train, col)
    excluye = ~((g["ic_inferior"] <= base_rate) & (g["ic_superior"] >= base_rate))
    print(f"{C.label(col):<24} {excluye.sum()}/{len(g)} categorías distinguibles"
          + (f"  ->  {', '.join(g.index[excluye])}" if excluye.any() else ""))

De 23 categorías repartidas en cinco variables, sólo cuatro tienen
intervalos que excluyen la tasa general:

| categoría | n | tasa | IC 95% |
|---|---|---|---|
| Género femenino | 4.272 | 17,5% | [16,4% – 18,7%] |
| Género masculino | 3.829 | 14,4% | [13,3% – 15,6%] |
| Nivel educativo: Doctorate | 356 | 20,5% | [16,3% – 24,7%] |
| Ingresos $60K–$80K | 1.136 | 13,8% | [11,8% – 15,8%] |

Las dos categorías de género se distinguen porque los grupos son grandes, no
porque la diferencia sea grande: 3,1 puntos porcentuales. Las otras dos apenas
rozan el borde del intervalo, y con cinco variables y 23 categorías examinadas,
encontrar dos casos marginales al 5% es lo que se espera por azar.

Lo importante es que **ninguna de estas diferencias resulta útil para
predecir**: en el notebook 02, la importancia por permutación deja al género
fuera de las diez primeras variables. Una diferencia puede ser
estadísticamente distinguible y a la vez inútil para ordenar clientes por
riesgo, y confundir esas dos cosas es un error frecuente al leer tablas como
esta.

La conclusión operativa se mantiene: la demografía no explica el abandono en
este dataset, y el análisis se concentra en el comportamiento transaccional. La
figura `reports/figures/01_tasa_por_categoria.png` muestra los intervalos
graficados junto al tamaño de cada grupo.

## 5. Variables derivadas

`src/features.py` construye cinco variables de intensidad transaccional
(ticket promedio, transacciones por mes, contactos por producto, ratio de
inactividad, caída de actividad) más un indicador de saldo rotativo en cero.

La motivación viene de la literatura de churn bancario: Brito et al. (2024)
encuentran que las variables de recencia, frecuencia y valor monetario superan
a las demográficas. El dataset no trae fechas, así que recencia no se puede
construir, pero sí intensidades normalizadas por antigüedad.

**El notebook 02 muestra que estas variables no mejoran la performance.** Se
documentan igual, con el resultado de la ablación.

In [ ]:
derivadas = ["ticket_promedio", "trans_por_mes", "contactos_por_producto",
             "ratio_inactividad", "caida_actividad", "saldo_rotativo_cero"]

comparacion = pd.DataFrame({
    "correlación con abandono": train[derivadas].corrwith(train[C.TARGET]).round(3),
    "media (se quedan)": train.loc[train[C.TARGET] == 0, derivadas].mean().round(2),
    "media (se van)": train.loc[train[C.TARGET] == 1, derivadas].mean().round(2),
})
comparacion.index = [C.label(i) for i in comparacion.index]
comparacion

## Qué se lleva el notebook 02

- `Avg_Open_To_Buy` y el índice de fila quedan fuera.
- Ninguna variable demográfica discrimina; la señal es transaccional.
- Ninguna correlación bivariada es fuerte, así que hay que probar modelos
  capaces de capturar interacciones.
- Accuracy queda descartada como métrica de selección; se usa PR-AUC.